<a href="https://colab.research.google.com/github/harjeet88/spark_examples/blob/master/llms/spark_faiss_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Step 1: Install Java 11 (required by Spark)
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null

# Step 2: Install PySpark + Spark NLP (latest compatible versions)
!pip install -q pyspark==3.5.3 spark-nlp==5.5.0

# Step 3: Set environment variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.8/620.8 kB 40.2 MB/s eta 0:00:00


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline

In [11]:
# Start Spark with required packages
spark = SparkSession.builder \
    .appName("TextEmbeddings") \
    .master("local[*]") \
    .config("spark.jars.packages",
            "com.johnsnowlabs.nlp:spark-nlp_2.12:5.5.0") \
    .config("spark.driver.memory", "16G") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

In [5]:
# Sample data (replace with your actual data)
data = [
    ("I love machine learning and PySpark",),
    ("Natural language processing is fascinating",),
    ("Spark is great for big data processing",),
    ("Sentence embeddings capture semantic meaning",),
]

In [6]:
df = spark.createDataFrame(data, ["text"])
df.show(truncate=False)

+--------------------------------------------+
|text                                        |
+--------------------------------------------+
|I love machine learning and PySpark         |
|Natural language processing is fascinating  |
|Spark is great for big data processing      |
|Sentence embeddings capture semantic meaning|
+--------------------------------------------+



In [7]:
# Define the embedding model (e.g., all-MiniLM-L6-v2 - fast & accurate)
DOCUMENT = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

In [8]:
SENTENCE = SentenceDetector() \
    .setInputCols(["document"]) \
    .setOutputCol("sentence")

In [17]:
embeddings = BertSentenceEmbeddings.pretrained("sent_biobert_pubmed_base_cased", "en") \
    .setInputCols(["sentence"]) \
    .setOutputCol("embeddings")

sent_biobert_pubmed_base_cased download started this may take some time.
Approximate size to download 386.4 MB
[OK!]


In [19]:
pipeline = Pipeline(stages=[DOCUMENT, SENTENCE, embeddings])
model = pipeline.fit(df)
result = model.transform(df)

In [32]:
from pyspark.sql.functions import element_at
# Extract embeddings as array<float> (taking the first and only element from the nested array)
final_df = result.select("text", element_at(col("embeddings.embeddings"), 1).alias("embedding_vector"))
final_df.show(truncate=60)

+--------------------------------------------+------------------------------------------------------------+
|                                        text|                                            embedding_vector|
+--------------------------------------------+------------------------------------------------------------+
|         I love machine learning and PySpark|[0.20513096, -0.008380315, 0.08129523, 0.19409728, -0.214...|
|  Natural language processing is fascinating|[2.2026304E-4, -0.0010675028, 0.14547028, 0.0041620494, 0...|
|      Spark is great for big data processing|[0.06280371, -0.098080665, 0.10823158, 0.087805815, 0.053...|
|Sentence embeddings capture semantic meaning|[-0.055992503, 0.11281232, 0.23320752, 0.21866654, -0.001...|
+--------------------------------------------+------------------------------------------------------------+



In [33]:
# Save to Parquet (in Colab folder)
final_df.write.mode("overwrite").parquet("/content/text_embeddings.parquet")
print("Saved! Open Files tab on the left to download text_embeddings.parquet")

Saved! Open Files tab on the left to download text_embeddings.parquet


In [34]:
parquet_path = "/content/text_embeddings.parquet"

In [25]:
!pip install -q autofaiss

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 48.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.

In [44]:
import pandas as pd
import numpy as np
import faiss
from pyspark.sql import SparkSession

In [45]:
# --- Configuration (Must match the generation step) ---
# Path where the embeddings were saved
PARQUET_PATH = parquet_path
# Dimension of the embeddings (e.g., all-MiniLM-L6-v2 is 384)
EMBEDDING_DIMENSION = 384
# Output path for the final FAISS index file
FAISS_INDEX_PATH = "faiss_index.bin"

In [46]:
# 1. Initialize Spark Session
print("Starting Spark Session...")
spark = SparkSession.builder \
    .appName("FAISSIndexing") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# 2. Read the Parquet file containing IDs and vectors
print(f"Reading embeddings from Parquet at: {PARQUET_PATH}...")
embeddings_df_spark = spark.read.parquet(PARQUET_PATH)

# Verify the schema
embeddings_df_spark.printSchema()


Starting Spark Session...
Reading embeddings from Parquet at: /content/text_embeddings.parquet...
root
 |-- text: string (nullable = true)
 |-- embedding_vector: array (nullable = true)
 |    |-- element: float (containsNull = true)



In [52]:
# 3. Collect Data to Driver and Convert to Pandas
# WARNING: The 'toPandas()' action collects ALL data onto the driver node.
# Ensure the driver has enough memory for your entire dataset.
print("Collecting data to Spark driver node (converting to Pandas)...")
# We only need 'embedding_vector' and 'text' if we want to retrieve the original text later
embeddings_pandas_df = embeddings_df_spark.select("text", "embedding_vector").toPandas()


In [53]:
# 4. Prepare Data for FAISS (NumPy conversion and ID generation)

# Convert the list/array column into a 2D NumPy matrix
# The 'embedding_vector' column contains Python lists of floats.
embeddings_matrix = np.stack(embeddings_pandas_df['embedding_vector'].to_numpy()).astype('float32')

# *** CRITICAL FIX: GENERATE SEQUENTIAL IDs ***
# Since the parquet file doesn't have an ID column, we use the row index
# (0, 1, 2, ...) as the unique ID for FAISS.
ids = np.arange(len(embeddings_pandas_df), dtype=np.int64)

print(f"Total vectors loaded: {embeddings_matrix.shape[0]}")
print(f"Vector dimension: {embeddings_matrix.shape[1]}")
print(f"Generated IDs range from 0 to {ids[-1]}")


Total vectors loaded: 4
Vector dimension: 768
Generated IDs range from 0 to 3


In [55]:
# 5. Build the FAISS Index
# We use IndexFlatL2 for exact L2 (Euclidean) distance search.
# This index structure stores all the vectors directly.
# The EMBEDDING_DIMENSION variable was incorrect; using the actual dimension from the embeddings_matrix.
index_flat = faiss.IndexFlatL2(embeddings_matrix.shape[1])


# We wrap it in IndexIDMap to store and retrieve the original 'row index' as the ID
index = faiss.IndexIDMap(index_flat)

# Add the vectors and their corresponding document IDs (row indices)
index.add_with_ids(embeddings_matrix, ids)

print(f"FAISS index built successfully with {index.ntotal} vectors.")

FAISS index built successfully with 4 vectors.


In [59]:
# 6. Save the FAISS Index to Disk
faiss.write_index(index, FAISS_INDEX_PATH)
print(f"FAISS index saved to disk at: {FAISS_INDEX_PATH}")

# --- Verification Search (Optional) ---
# Load the index and perform a test search
print("\n--- Performing a test search ---")
index_loaded = faiss.read_index(FAISS_INDEX_PATH)

# Use the vector corresponding to Row Index 0 as the query vector
query_id_index = ids[2] # Which is 0
query_vector = embeddings_matrix[query_id_index].reshape(1, -1)
k = 3 # Search for the 3 nearest neighbors


FAISS index saved to disk at: faiss_index.bin

--- Performing a test search ---


In [60]:
D, I = index_loaded.search(query_vector, k)

# The closest neighbor (k=1) should be the vector itself (ID 0) with distance D=0
print(f"Query ID (Row Index): {query_id_index}")
print(f"Distances (D): {D[0]}")
print(f"Nearest Document IDs (Row Indices): {I[0]}")

# --- Detailed Search Results ---
query_text = embeddings_pandas_df.iloc[query_id_index]['text']
print("\n--- Detailed Search Results ---")
print(f"Query Text: '{query_text}'\n")

# Iterate through the returned indices (I) and distances (D)
for rank, (doc_id, dist) in enumerate(zip(I[0], D[0])):
    retrieved_text = embeddings_pandas_df.iloc[doc_id]['text']
    print(f"Rank {rank+1} (ID: {doc_id})")
    print(f"  Distance: {dist:.4f}")


Query ID (Row Index): 2
Distances (D): [ 0.       22.004276 30.100533]
Nearest Document IDs (Row Indices): [2 0 3]

--- Detailed Search Results ---
Query Text: 'I love machine learning and PySpark'

Rank 1 (ID: 2)
  Distance: 0.0000
Rank 2 (ID: 0)
  Distance: 22.0043
Rank 3 (ID: 3)
  Distance: 30.1005


In [61]:
# --- Detailed Search Results ---
query_text = embeddings_pandas_df.iloc[query_id_index]['text']
print("\n--- Detailed Search Results ---")
print(f"Query Text: '{query_text}'\n")

# Iterate through the returned indices (I) and distances (D)
for rank, (doc_id, dist) in enumerate(zip(I[0], D[0])):
    retrieved_text = embeddings_pandas_df.iloc[doc_id]['text']
    print(f"Rank {rank+1} (ID: {doc_id})")
    print(f"  Distance: {dist:.4f}")
    print(f"  Text: '{retrieved_text}'")


--- Detailed Search Results ---
Query Text: 'I love machine learning and PySpark'

Rank 1 (ID: 2)
  Distance: 0.0000
  Text: 'I love machine learning and PySpark'
Rank 2 (ID: 0)
  Distance: 22.0043
  Text: 'Spark is great for big data processing'
Rank 3 (ID: 3)
  Distance: 30.1005
  Text: 'Natural language processing is fascinating'
